# Get images from [here](https://www.gbif.org)

In [2]:
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
import mimetypes

BASE_URL = "https://api.gbif.org/v1/occurrence/search"

SPECIES = [
    "Agalychnis callidryas",
    "Breviceps gibbosus",
    "Ceratophrys cornuta",
    "Dendrobates tinctorius",
    "Hyalinobatrachium fleischmanni"
]

TARGET_DIR = Path("../dataset/")
MAX_IMAGES = 1000
BATCH_SIZE = 300
MAX_WORKERS = 10

session = requests.Session()


def get_extension(response, url):
    content_type = response.headers.get("Content-Type", "").lower()

    # если сервер честный — берём из заголовка
    if "image" in content_type:
        ext = mimetypes.guess_extension(content_type.split(";")[0])
        if ext:
            return ext

    # fallback — пробуем взять из URL
    path = urlparse(url).path
    suffix = Path(path).suffix
    if suffix:
        return suffix

    return None  # неизвестный тип


def download_image(url, dir_path, index):
    try:
        r = session.get(url, timeout=10, stream=True)

        if r.status_code != 200:
            return False

        ext = get_extension(r, url)
        if not ext:
            return False  # не картинка

        filename = dir_path / f"{index}{ext}"

        # если файл уже есть — пропускаем
        if filename.exists():
            return False

        with open(filename, "wb") as f:
            for chunk in r.iter_content(8192):  # потоковая запись
                if chunk:
                    f.write(chunk)

        return True

    except requests.RequestException:
        return False


for class_id, species in enumerate(SPECIES, start=1):

    dir_path = TARGET_DIR / f"class{class_id}"
    dir_path.mkdir(parents=True, exist_ok=True)

    image_urls = []
    offset = 0

    while len(image_urls) < MAX_IMAGES:

        params = {
            "scientificName": species,
            "mediaType": "StillImage",
            "limit": BATCH_SIZE,
            "offset": offset
        }

        r = session.get(BASE_URL, params=params, timeout=10)
        if r.status_code != 200:
            break

        data = r.json()
        results = data.get("results", [])
        if not results:
            break

        for record in results:
            for m in record.get("media", []):
                url = m.get("identifier")
                if url:
                    image_urls.append(url)
                    if len(image_urls) >= MAX_IMAGES:
                        break
            if len(image_urls) >= MAX_IMAGES:
                break

        offset += BATCH_SIZE

    print(f"Collected URLs ({class_id}):", len(image_urls))

    downloaded = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(download_image, url, dir_path, idx)
            for idx, url in enumerate(image_urls)
        ]

        for future in as_completed(futures):
            if future.result():
                downloaded += 1

    print(f"Downloaded ({class_id}):", downloaded)

Collected URLs (1): 1000
Downloaded (1): 998
Collected URLs (2): 374
Downloaded (2): 371
Collected URLs (3): 726
Downloaded (3): 718
Collected URLs (4): 398
Downloaded (4): 387
Collected URLs (5): 285
Downloaded (5): 241
